In [1]:
import pickle
import faiss
import os
import json
from dotenv import load_dotenv
from google import genai

# Load corpus
with open("../data/all_chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

index = faiss.read_index("../data/all_index.faiss")

# Gemini client
load_dotenv("../.env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loaded:", len(all_chunks), "chunks")
print("Index size:", index.ntotal)

Loaded: 1356 chunks
Index size: 1356


In [2]:
def extract_info(pdf_name):
    """Extract task/method/dataset/results as structured JSON."""
    paper_chunks = [c["text"] for c in all_chunks if c["source"] == pdf_name]
    if not paper_chunks:
        return None

    # Sample chunks across the paper (first, middle, last) to capture full content
    n = len(paper_chunks)
    sample_indices = list(range(0, min(10, n))) + \
                     list(range(n//2 - 5, min(n//2 + 5, n))) + \
                     list(range(max(0, n-10), n))
    sample_indices = sorted(set(sample_indices))
    context = "\n\n".join(paper_chunks[i] for i in sample_indices)

    prompt = f"""Extract structured information from this research paper.
Return ONLY valid JSON, no explanation, no markdown fences.

Format:
{{
  "task": "what problem the paper addresses (1 sentence)",
  "method": "name and short description of the approach (1-2 sentences)",
  "dataset": "datasets used for evaluation, or 'Not specified'",
  "results": "key quantitative results (1-2 sentences)"
}}

Paper text:
{context}

JSON:"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    text = response.text.strip()

    # Strip markdown fences if Gemini added them
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {pdf_name}: {e}")
        print("Raw:", text[:300])
        return None

# Test on one paper
info = extract_info("2506.06962v3.pdf")
print(json.dumps(info, indent=2))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "task": "The paper addresses local distortions, inconsistencies, and static retrieval limitations like over-copying and stylistic bias in retrieval-augmented image generation.",
  "method": "Autoregressive Retrieval Augmentation (AR-RAG) incorporates k-nearest neighbor retrievals at the patch level during decoding via two frameworks: Distribution-Augmentation in Decoding (DAiD), a training-free distribution-merging strategy, and Feature-Augmentation in Decoding (FAiD), a parameter-efficient fine-tuning approach with multi-scale feature smoothing.",
  "dataset": "Midjourney-30K, GenEval, DPG-Bench, and Midjourney-10K",
  "results": "AR-RAG achieves significant performance gains over state-of-the-art baselines on key benchmarks, with DAiD and FAiD obtaining optimal FID scores of 14.12 and 13.13, respectively, on Midjourney-10K."
}


In [3]:
# Pick papers to compare 
papers_to_compare = [
    "2506.06962v3.pdf",   # AR-RAG
    "2601.07528v2.pdf",   # agentic Quran grounding
    "2507.23334v2.pdf",   # MusT-RAG
    "2510.25621v1.pdf",   # RAG evolution
]

extracted = {}
for paper in papers_to_compare:
    print(f"Extracting: {paper} ...")
    info = extract_info(paper)
    if info:
        extracted[paper] = info
    else:
        print(f"  Failed on {paper}")

print(f"\nExtracted info from {len(extracted)} papers")

Extracting: 2506.06962v3.pdf ...
Extracting: 2601.07528v2.pdf ...
Extracting: 2507.23334v2.pdf ...
Extracting: 2510.25621v1.pdf ...

Extracted info from 4 papers


In [4]:
import pandas as pd

# Build a table: rows = papers, columns = task/method/dataset/results
rows = []
for paper, info in extracted.items():
    rows.append({
        "Paper": paper,
        "Task": info.get("task", ""),
        "Method": info.get("method", ""),
        "Dataset": info.get("dataset", ""),
        "Results": info.get("results", ""),
    })

df_compare = pd.DataFrame(rows)

# Truncate long cells for display
pd.set_option("display.max_colwidth", 80)

df_compare

,Paper,Task,Method,Dataset,Results
0,2506.06962v3.pdf,"The paper addresses the issues of local distortions, inconsistencies, and la...",Autoregressive Retrieval Augmentation (AR-RAG) autoregressively incorporates...,"Midjourney-30K, Midjourney-10K, GenEval, and DPG-Bench","On Midjourney-10K, DAiD and FAiD achieve FID scores of 14.12 and 13.13 respe..."
1,2601.07528v2.pdf,"The paper addresses ungrounded responses, free-form hallucinations, and the ...","Agentic RAG, an agentic Quran-grounding framework that uses structured tool ...","ISLAMICFAITHQA, IslamicEval 2025 Shared Task 2",The dense retriever mE5-base achieved a MAP_Q@5 of 0.138 compared to 0.094 f...
2,2507.23334v2.pdf,Adapting general-purpose large language models for text-only music question ...,"MusT-RAG, a retrieval-augmented generation framework that uses a specialized...",ArtistMus and TrustMus,MusT-RAG with RAG fine-tuning achieved 82.4% on factual questions and 92.0% ...
3,2510.25621v1.pdf,"The paper addresses hallucination, lack of faithfulness, and reasoning over ...","FARSIQA, built on the FAIR-RAG framework, employs an adaptive, iterative ref...",IslamicPCQA benchmark,The system achieves a 97.0% Negative Rejection score (a 40-point improvement...


In [5]:
def synthesize_comparison(extracted):
    # Build a text description of all papers for the prompt
    paper_text = ""
    for paper, info in extracted.items():
        paper_text += f"""Paper: {paper}
Task: {info.get('task','')}
Method: {info.get('method','')}
Dataset: {info.get('dataset','')}
Results: {info.get('results','')}

"""

    prompt = f"""You are analyzing {len(extracted)} research papers. Write a comparison
synthesis (4-6 sentences) that answers:

1. What do these papers have in common (shared problem, approach, or theme)?
2. How do they differ in method or scope?
3. Which approaches seem most related to each other?

Be specific. Reference papers by their topic (e.g., "the image generation paper"),
not by filename. Use only the info below.

{paper_text}

Synthesis:"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )
    return response.text

synthesis = synthesize_comparison(extracted)
print(synthesis)

All four papers share the core theme of leveraging retrieval-augmented generation (RAG) to overcome limitations in generative models, specifically targeting hallucinations, domain adaptation challenges, and output distortions. They differ substantially in modality and scope: the image generation paper applies dynamic, patch-level retrieval to visual decoding via training-free distribution merging and feature smoothing, whereas the other three papers focus exclusively on text-based question answering within specialized domains. Methodologically, the music question answering paper integrates a specialized vector database across both fine-tuning and inference stages, while the religious question answering papers focus on iterative evidence retrieval. As a result, the two Islamic question answering papers are the most closely related to each other. Both frameworks address ungrounded religious responses through iterative refinement processes—one using structured tool calls for evidence seek